# 🌞 Solar Power Plant: Generation Forecasting & Anomaly Detection

## Problem Statement
This project tackles two interrelated objectives:
1. **Regression** – Predict AC power generation at the inverter level using weather sensor data (irradiance, temperature, humidity, wind speed).
2. **Anomaly Detection** – Flag underperforming inverters by comparing predicted vs. actual yields, identifying potential panel faults, soiling, or shading issues.

### Datasets Used
- `Plant_1_Generation_Data.csv` – Per-inverter DC/AC power output (22 inverters, May–Jun 2020)
- `Plant_2_Generation_Data.csv` – Same structure, second plant
- `SolarRecording.csv` – Weather sensor data (radiation, temperature, humidity, wind)

---

## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, TimeSeriesSplit, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, IsolationForest
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.pipeline import Pipeline

# SHAP for model explainability (pip install shap if needed)
try:
    import shap
    SHAP_AVAILABLE = True
    print("SHAP available ✅")
except ImportError:
    SHAP_AVAILABLE = False
    print("SHAP not installed. Run: pip install shap  (optional but recommended)")

# Plotting style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (12, 5)

print('All core libraries loaded successfully.')

---
## 2. Data Loading & Initial Inspection

In [ ]:
# Load datasets
p1 = pd.read_csv('Plant_1_Generation_Data.csv')
p2 = pd.read_csv('Plant_2_Generation_Data.csv')
sr = pd.read_csv('SolarRecording.csv')

print('=== Plant 1 Generation Data ===')
print(f'Shape: {p1.shape}  |  Inverters: {p1["SOURCE_KEY"].nunique()}')
display(p1.head(3))

print('\n=== Plant 2 Generation Data ===')
print(f'Shape: {p2.shape}  |  Inverters: {p2["SOURCE_KEY"].nunique()}')
display(p2.head(3))

print('\n=== Solar Weather Recording ===')
print(f'Shape: {sr.shape}')
display(sr.head(3))

In [ ]:
print('Plant 1 dtypes:'); print(p1.dtypes)
print('\nPlant 2 dtypes:'); print(p2.dtypes)
print('\nSolar Recording dtypes:'); print(sr.dtypes)

In [ ]:
print('Missing values:')
print('Plant 1:', p1.isnull().sum().sum())
print('Plant 2:', p2.isnull().sum().sum())
print('Solar Recording:', sr.isnull().sum().sum())

---
## 3. Data Preprocessing & Feature Engineering

In [ ]:
# ---- Parse datetimes ----
# Plant 1 uses DD-MM-YYYY format
p1['DATE_TIME'] = pd.to_datetime(p1['DATE_TIME'], dayfirst=True)
# Plant 2 uses standard ISO format
p2['DATE_TIME'] = pd.to_datetime(p2['DATE_TIME'])
# Solar Recording: combine Date + Time
sr['datetime'] = pd.to_datetime(sr['Date'] + ' ' + sr['Time'], format='%m/%d/%Y %H:%M:%S')

print('Plant 1 range:', p1['DATE_TIME'].min(), '->', p1['DATE_TIME'].max())
print('Plant 2 range:', p2['DATE_TIME'].min(), '->', p2['DATE_TIME'].max())
print('SR range:     ', sr['datetime'].min(), '->', sr['datetime'].max())

In [ ]:
# ---- Resample Solar Recording to 15-minute intervals to match plant data ----
sr = sr.drop(columns=['Unnamed: 0'], errors='ignore')
sr_indexed = sr.set_index('datetime').sort_index()
weather_num_cols = ['Radiation', 'Temperature', 'Pressure', 'Humidity', 'WindDirection(Degrees)', 'Speed']
sr_15min = sr_indexed[weather_num_cols].resample('15min').mean().interpolate(method='linear')
sr_15min = sr_15min.reset_index()
sr_15min.columns = ['DATE_TIME'] + weather_num_cols
print('Resampled weather data shape:', sr_15min.shape)
sr_15min.head()

In [ ]:
# ---- Aggregate plant data to plant level (sum across inverters per timestamp) ----
def aggregate_plant(df, plant_id_label):
    agg = df.groupby('DATE_TIME').agg(
        DC_POWER=('DC_POWER', 'sum'),
        AC_POWER=('AC_POWER', 'sum'),
        DAILY_YIELD=('DAILY_YIELD', 'mean')
    ).reset_index()
    agg['PLANT'] = plant_id_label
    return agg

p1_agg = aggregate_plant(p1, 'Plant_1')
p2_agg = aggregate_plant(p2, 'Plant_2')

print('Plant 1 aggregated:', p1_agg.shape)
print('Plant 2 aggregated:', p2_agg.shape)

In [ ]:
# ---- Merge plant data with weather data ----
def merge_with_weather(plant_agg, weather_df):
    merged = pd.merge_asof(
        plant_agg.sort_values('DATE_TIME'),
        weather_df.sort_values('DATE_TIME'),
        on='DATE_TIME',
        tolerance=pd.Timedelta('15min'),
        direction='nearest'
    )
    merged = merged.dropna(subset=['Radiation'])
    return merged

p1_merged = merge_with_weather(p1_agg, sr_15min)
p2_merged = merge_with_weather(p2_agg, sr_15min)

print('Plant 1 merged shape:', p1_merged.shape)
print('Plant 2 merged shape:', p2_merged.shape)
display(p1_merged.head())

In [ ]:
# ---- Feature Engineering ----
def add_time_features(df):
    df = df.copy()
    df['hour'] = df['DATE_TIME'].dt.hour
    df['month'] = df['DATE_TIME'].dt.month
    df['dayofyear'] = df['DATE_TIME'].dt.dayofyear
    # Solar hour angle proxy (how far from solar noon)
    df['hour_from_noon'] = (df['hour'] - 12).abs()
    # Remove nighttime rows (no generation possible)
    df = df[df['Radiation'] > 10].copy()
    return df

p1_feat = add_time_features(p1_merged)
p2_feat = add_time_features(p2_merged)

print('Plant 1 daytime rows:', p1_feat.shape[0])
print('Plant 2 daytime rows:', p2_feat.shape[0])

---
## 4. Exploratory Data Analysis

In [ ]:
# ---- Daily AC Power Generation over time ----
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=False)

for ax, df, label, color in zip(axes, [p1_merged, p2_merged], ['Plant 1', 'Plant 2'], ['steelblue', 'darkorange']):
    daily = df.groupby(df['DATE_TIME'].dt.date)['AC_POWER'].sum() / 1000  # kW
    ax.fill_between(range(len(daily)), daily.values, alpha=0.4, color=color)
    ax.plot(range(len(daily)), daily.values, color=color, linewidth=1.5)
    ax.set_title(f'{label} – Daily Total AC Power Output', fontsize=13)
    ax.set_ylabel('AC Power (kW)')
    ax.set_xticks(range(len(daily)))
    ax.set_xticklabels([str(d) for d in daily.index], rotation=45, ha='right', fontsize=7)

plt.tight_layout()
plt.suptitle('Daily Power Generation Comparison', y=1.02, fontsize=14, fontweight='bold')
plt.show()

In [ ]:
# ---- Average AC Power by Hour of Day (both plants) ----
fig, ax = plt.subplots(figsize=(10, 4))

for df, label, color in zip([p1_merged, p2_merged], ['Plant 1', 'Plant 2'], ['steelblue', 'darkorange']):
    hourly = df.groupby(df['DATE_TIME'].dt.hour)['AC_POWER'].mean()
    ax.plot(hourly.index, hourly.values, marker='o', label=label, color=color, linewidth=2)

ax.set_xlabel('Hour of Day')
ax.set_ylabel('Mean AC Power')
ax.set_title('Average AC Power Output by Hour of Day', fontsize=13)
ax.legend()
ax.set_xticks(range(0, 24))
plt.tight_layout()
plt.show()

In [ ]:
# ---- Correlation heatmap: weather features vs AC Power (Plant 1) ----
corr_cols = ['Radiation', 'Temperature', 'Humidity', 'Speed', 'AC_POWER']
corr_data = p1_feat[corr_cols].corr()

plt.figure(figsize=(7, 5))
sns.heatmap(corr_data, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            linewidths=0.5, square=True)
plt.title('Correlation Heatmap – Plant 1 (Daytime Only)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ---- Radiation vs AC Power scatter ----
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, df, label, color in zip(axes, [p1_feat, p2_feat], ['Plant 1', 'Plant 2'], ['steelblue', 'darkorange']):
    ax.scatter(df['Radiation'], df['AC_POWER'], alpha=0.2, s=10, color=color)
    ax.set_xlabel('Radiation (W/m²)')
    ax.set_ylabel('AC Power')
    ax.set_title(f'{label}: Radiation vs AC Power')

plt.suptitle('Solar Irradiance vs Power Generation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ---- Per-inverter daily yield distribution (Plant 1) ----
p1['DATE_TIME'] = pd.to_datetime(p1['DATE_TIME'], dayfirst=True)
p1_daily_inv = p1.groupby(['SOURCE_KEY', p1['DATE_TIME'].dt.date])['DAILY_YIELD'].max().reset_index()
p1_daily_inv.columns = ['SOURCE_KEY', 'Date', 'DAILY_YIELD']

plt.figure(figsize=(14, 5))
inv_order = p1_daily_inv.groupby('SOURCE_KEY')['DAILY_YIELD'].median().sort_values()
sns.boxplot(data=p1_daily_inv, x='SOURCE_KEY', y='DAILY_YIELD', order=inv_order.index, palette='Blues')
plt.xticks(rotation=90, fontsize=8)
plt.title('Plant 1 – Daily Yield Distribution per Inverter', fontsize=13)
plt.xlabel('Inverter (SOURCE_KEY)')
plt.ylabel('Daily Yield')
plt.tight_layout()
plt.show()
print('Note: Inverters on the left have consistently lower yields – potential underperformers!')

In [ ]:
# ---- DC vs AC efficiency per inverter (Plant 1) ----
p1_inv_stats = p1.groupby('SOURCE_KEY').agg(
    mean_DC=('DC_POWER', 'mean'),
    mean_AC=('AC_POWER', 'mean')
).reset_index()
p1_inv_stats['efficiency'] = p1_inv_stats['mean_AC'] / (p1_inv_stats['mean_DC'] + 1e-9)

plt.figure(figsize=(14, 4))
colors = ['red' if e < p1_inv_stats['efficiency'].quantile(0.25) else 'steelblue' for e in p1_inv_stats['efficiency']]
plt.bar(p1_inv_stats['SOURCE_KEY'], p1_inv_stats['efficiency'], color=colors)
plt.axhline(p1_inv_stats['efficiency'].mean(), color='orange', linestyle='--', label=f'Mean efficiency: {p1_inv_stats["efficiency"].mean():.3f}')
plt.xticks(rotation=90, fontsize=7)
plt.title('Plant 1 – DC→AC Conversion Efficiency per Inverter (Red = Low)', fontsize=13)
plt.ylabel('AC/DC Ratio')
plt.legend()
plt.tight_layout()
plt.show()

### Inverter × Date Performance Heatmap
Shows *when* each inverter underperforms — helps distinguish one-off events (weather) from persistent faults (hardware).

In [ ]:
# ---- Inverter × Date Performance Heatmap (Plant 1) ----
# Build daily yield per inverter
p1['DATE_TIME'] = pd.to_datetime(p1['DATE_TIME'], dayfirst=True)
p1_heatmap = p1.groupby(['SOURCE_KEY', p1['DATE_TIME'].dt.date])['DAILY_YIELD'].max().reset_index()
p1_heatmap.columns = ['SOURCE_KEY', 'date', 'DAILY_YIELD']

# Pivot to matrix: rows=inverter, cols=date
heatmap_pivot = p1_heatmap.pivot_table(index='SOURCE_KEY', columns='date', values='DAILY_YIELD')

# Normalise each row so inverters are comparable (relative to their own max)
heatmap_norm = heatmap_pivot.div(heatmap_pivot.max(axis=1), axis=0)

plt.figure(figsize=(18, 8))
sns.heatmap(heatmap_norm, cmap='RdYlGn', linewidths=0.3, linecolor='gray',
            cbar_kws={'label': 'Normalised Daily Yield (1.0 = personal best)'},
            vmin=0, vmax=1)
plt.title('Plant 1 – Per-Inverter Daily Performance Heatmap\n(Red = underperforming relative to own best)', fontsize=13)
plt.xlabel('Date')
plt.ylabel('Inverter (SOURCE_KEY)')
plt.xticks(rotation=45, ha='right', fontsize=7)
plt.yticks(fontsize=7)
plt.tight_layout()
plt.show()
print("Columns of persistent red = hardware/degradation issue. Rows of red = weather event affecting all inverters.")

In [ ]:
# ---- Plant 2 performance heatmap ----
p2_heatmap = p2.groupby(['SOURCE_KEY', p2['DATE_TIME'].dt.date])['DAILY_YIELD'].max().reset_index()
p2_heatmap.columns = ['SOURCE_KEY', 'date', 'DAILY_YIELD']
heatmap2_pivot = p2_heatmap.pivot_table(index='SOURCE_KEY', columns='date', values='DAILY_YIELD')
heatmap2_norm = heatmap2_pivot.div(heatmap2_pivot.max(axis=1), axis=0)

plt.figure(figsize=(18, 8))
sns.heatmap(heatmap2_norm, cmap='RdYlGn', linewidths=0.3, linecolor='gray',
            cbar_kws={'label': 'Normalised Daily Yield'}, vmin=0, vmax=1)
plt.title('Plant 2 – Per-Inverter Daily Performance Heatmap', fontsize=13)
plt.xlabel('Date')
plt.ylabel('Inverter (SOURCE_KEY)')
plt.xticks(rotation=45, ha='right', fontsize=7)
plt.yticks(fontsize=7)
plt.tight_layout()
plt.show()

---
## 5. Regression – Predicting AC Power from Weather Features

We train on Plant 1 data and test both within Plant 1 and across Plant 2 (cross-plant generalization).

In [ ]:
FEATURES = ['Radiation', 'Temperature', 'Humidity', 'Speed', 'hour', 'hour_from_noon', 'month']
TARGET = 'AC_POWER'

# Plant 1 train/test split
X1 = p1_feat[FEATURES]
y1 = p1_feat[TARGET]
X1_train, X1_test, y1_train, y1_test = train_test_split(X1, y1, test_size=0.2, random_state=42)

# Plant 2 for cross-plant test
X2 = p2_feat[FEATURES]
y2 = p2_feat[TARGET]

print(f'Plant 1 Train: {X1_train.shape[0]} | Test: {X1_test.shape[0]}')
print(f'Plant 2 (cross-plant eval): {X2.shape[0]}')

In [ ]:
# ---- Define and train models ----
models = {
    'Linear Regression': Pipeline([('scaler', StandardScaler()), ('model', LinearRegression())]),
    'Ridge Regression': Pipeline([('scaler', StandardScaler()), ('model', Ridge(alpha=1.0))]),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
}

results = []

for name, model in models.items():
    model.fit(X1_train, y1_train)
    # Evaluate on Plant 1 test
    y_pred_1 = model.predict(X1_test)
    r2_1 = r2_score(y1_test, y_pred_1)
    rmse_1 = np.sqrt(mean_squared_error(y1_test, y_pred_1))
    mae_1 = mean_absolute_error(y1_test, y_pred_1)
    # Evaluate on Plant 2 (cross-plant)
    y_pred_2 = model.predict(X2)
    r2_2 = r2_score(y2, y_pred_2)
    rmse_2 = np.sqrt(mean_squared_error(y2, y_pred_2))
    results.append({'Model': name, 'R² (P1 test)': r2_1, 'RMSE (P1 test)': rmse_1, 'MAE (P1 test)': mae_1,
                    'R² (P2 cross)': r2_2, 'RMSE (P2 cross)': rmse_2})
    print(f'{name:25s} | R²={r2_1:.4f} | RMSE={rmse_1:.1f} | [P2] R²={r2_2:.4f}')

results_df = pd.DataFrame(results)
display(results_df.round(4))

In [ ]:
# ---- Model Comparison Bar Chart ----
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']
axes[0].bar(results_df['Model'], results_df['R² (P1 test)'], color=colors)
axes[0].set_ylim(0, 1)
axes[0].set_title('R² Score – Plant 1 Test Set')
axes[0].set_xticklabels(results_df['Model'], rotation=20, ha='right')
axes[0].set_ylabel('R²')

axes[1].bar(results_df['Model'], results_df['RMSE (P1 test)'], color=colors)
axes[1].set_title('RMSE – Plant 1 Test Set')
axes[1].set_xticklabels(results_df['Model'], rotation=20, ha='right')
axes[1].set_ylabel('RMSE')

plt.suptitle('Regression Model Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 5b. Time-Series Cross-Validation
Standard `train_test_split` shuffles randomly — this leaks future data into training, which is incorrect for time-series. `TimeSeriesSplit` respects temporal order and gives a more honest estimate of real-world performance.

In [ ]:
# ---- TimeSeriesSplit cross-validation ----
from sklearn.model_selection import cross_val_score

# Sort by time (critical for TSCV)
p1_feat_sorted = p1_feat.sort_values('DATE_TIME').reset_index(drop=True)
X_ts = p1_feat_sorted[FEATURES]
y_ts = p1_feat_sorted[TARGET]

tscv = TimeSeriesSplit(n_splits=5)

print("TimeSeriesSplit CV Results (R²) — respects temporal order:")
print(f"{'Model':<25} {'Fold 1':>8} {'Fold 2':>8} {'Fold 3':>8} {'Fold 4':>8} {'Fold 5':>8} {'Mean':>8} {'Std':>7}")
print("-" * 85)

cv_results = {}
for name, model in models.items():
    scores = cross_val_score(model, X_ts, y_ts, cv=tscv, scoring='r2', n_jobs=-1)
    cv_results[name] = scores
    fold_str = "  ".join([f"{s:>7.4f}" for s in scores])
    print(f"{name:<25} {fold_str}  {scores.mean():>7.4f}  {scores.std():>6.4f}")

# Visualise CV score distributions
fig, ax = plt.subplots(figsize=(9, 5))
cv_df = pd.DataFrame(cv_results)
cv_df.boxplot(ax=ax)
ax.axhline(0, color='red', linestyle='--', alpha=0.5)
ax.set_title('TimeSeriesSplit CV – R² Score Distribution per Model', fontsize=13)
ax.set_ylabel('R² Score')
ax.set_xticklabels(list(cv_results.keys()), rotation=15, ha='right')
plt.tight_layout()
plt.show()

### 5c. Hyperparameter Tuning – Random Forest
Using `RandomizedSearchCV` with `TimeSeriesSplit` to find the best Random Forest configuration without overfitting.

In [ ]:
# ---- RandomizedSearchCV for Random Forest with TimeSeriesSplit ----
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', 0.5]
}

rf_base = RandomForestRegressor(random_state=42, n_jobs=-1)
tscv_tune = TimeSeriesSplit(n_splits=3)

random_search = RandomizedSearchCV(
    rf_base, param_distributions=param_dist,
    n_iter=20, cv=tscv_tune, scoring='r2',
    random_state=42, n_jobs=-1, verbose=1
)
random_search.fit(X_ts, y_ts)

print("\nBest parameters found:")
for k, v in random_search.best_params_.items():
    print(f"  {k}: {v}")
print(f"\nBest CV R²: {random_search.best_score_:.4f}")

# Replace best model with tuned version
best_tuned_rf = random_search.best_estimator_
y_pred_tuned = best_tuned_rf.predict(X1_test)
print(f"Tuned RF on holdout test: R²={r2_score(y1_test, y_pred_tuned):.4f}, RMSE={np.sqrt(mean_squared_error(y1_test, y_pred_tuned)):.1f}")

# Compare default vs tuned
default_rf = models['Random Forest']
y_pred_default = default_rf.predict(X1_test)
print(f"Default RF on holdout test: R²={r2_score(y1_test, y_pred_default):.4f}")

### 5d. SHAP – Model Explainability (Beyond the Course)
SHAP (SHapley Additive exPlanations) shows not just *which* features matter but *how* they push predictions up or down — critical for convincing engineers which weather conditions drive losses.

In [ ]:
# ---- SHAP values for the tuned Random Forest ----
if SHAP_AVAILABLE:
    # Use a sample for speed
    X_sample = X1_test.sample(min(500, len(X1_test)), random_state=42)
    
    explainer = shap.TreeExplainer(best_tuned_rf)
    shap_values = explainer.shap_values(X_sample)
    
    # Summary plot (beeswarm — shows direction AND magnitude)
    plt.figure(figsize=(9, 6))
    shap.summary_plot(shap_values, X_sample, feature_names=FEATURES, show=False)
    plt.title('SHAP Summary Plot – Feature Impact on AC Power Prediction', fontsize=12, pad=20)
    plt.tight_layout()
    plt.show()
    
    # Bar summary (mean absolute SHAP)
    plt.figure(figsize=(8, 5))
    shap.summary_plot(shap_values, X_sample, feature_names=FEATURES, plot_type='bar', show=False)
    plt.title('Mean |SHAP| – Feature Importance', fontsize=12)
    plt.tight_layout()
    plt.show()
    
    print("\nKey insight: Radiation drives predictions UP at high values.")
    print("High humidity tends to push predictions DOWN (cloud cover proxy).")
else:
    print("Install SHAP to run this cell: pip install shap")
    print("SHAP provides directional feature importance — key for the presentation 'something new' section.")

In [ ]:
# ---- Best model: Predicted vs Actual ----
# Use tuned model from RandomizedSearchCV
y_pred_best = best_tuned_rf.predict(X1_test)

plt.figure(figsize=(7, 7))
plt.scatter(y1_test, y_pred_best, alpha=0.3, s=12, color='steelblue')
lims = [0, max(y1_test.max(), y_pred_best.max())]
plt.plot(lims, lims, 'r--', linewidth=2, label='Perfect Prediction')
plt.xlabel('Actual AC Power')
plt.ylabel('Predicted AC Power')
plt.title(f'Random Forest – Predicted vs Actual\nR²={r2_score(y1_test, y_pred_best):.4f}', fontsize=12)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ---- Feature Importance (Random Forest) ----
rf_model = best_model.named_steps['model'] if hasattr(best_model, 'named_steps') else best_model
importances = rf_model.feature_importances_
feat_imp = pd.Series(importances, index=FEATURES).sort_values(ascending=True)

plt.figure(figsize=(8, 5))
feat_imp.plot(kind='barh', color='steelblue')
plt.title('Random Forest – Feature Importances', fontsize=13)
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

---
## 6. Anomaly Detection – Flagging Underperforming Inverters

### Approach
For each inverter in Plant 1:
1. Use the best regression model to predict expected AC power based on weather conditions.
2. Compute the **residual** = actual − predicted.
3. Apply **Isolation Forest** on per-inverter daily performance metrics to flag anomalies.
4. Rank inverters by anomaly severity.

In [ ]:
# ---- Per-inverter level merge with weather ----
p1['DATE_TIME'] = pd.to_datetime(p1['DATE_TIME'], dayfirst=True)
p1_inv_weather = pd.merge_asof(
    p1.sort_values('DATE_TIME'),
    sr_15min.sort_values('DATE_TIME'),
    on='DATE_TIME',
    tolerance=pd.Timedelta('15min'),
    direction='nearest'
).dropna(subset=['Radiation'])

p1_inv_weather['hour'] = p1_inv_weather['DATE_TIME'].dt.hour
p1_inv_weather['month'] = p1_inv_weather['DATE_TIME'].dt.month
p1_inv_weather['hour_from_noon'] = (p1_inv_weather['hour'] - 12).abs()

# Filter daytime
p1_inv_day = p1_inv_weather[p1_inv_weather['Radiation'] > 10].copy()

# Predict expected AC power per row using the trained model
p1_inv_day['AC_PREDICTED'] = best_model.predict(p1_inv_day[FEATURES])
p1_inv_day['RESIDUAL'] = p1_inv_day['AC_POWER'] - p1_inv_day['AC_PREDICTED']
p1_inv_day['RESIDUAL_PCT'] = p1_inv_day['RESIDUAL'] / (p1_inv_day['AC_PREDICTED'].abs() + 1)

print('Per-inverter prediction done. Sample:')
display(p1_inv_day[['DATE_TIME', 'SOURCE_KEY', 'AC_POWER', 'AC_PREDICTED', 'RESIDUAL', 'RESIDUAL_PCT']].head())

In [ ]:
# ---- Daily performance statistics per inverter ----
p1_inv_day['date'] = p1_inv_day['DATE_TIME'].dt.date

inv_daily = p1_inv_day.groupby(['SOURCE_KEY', 'date']).agg(
    mean_residual=('RESIDUAL', 'mean'),
    mean_residual_pct=('RESIDUAL_PCT', 'mean'),
    total_AC=('AC_POWER', 'sum'),
    total_AC_pred=('AC_PREDICTED', 'sum'),
    std_residual=('RESIDUAL', 'std')
).reset_index()
inv_daily['performance_ratio'] = inv_daily['total_AC'] / (inv_daily['total_AC_pred'] + 1)

print('Inverter daily performance table shape:', inv_daily.shape)
display(inv_daily.head())

In [ ]:
# ---- Isolation Forest Anomaly Detection on inverter-level features ----
anomaly_features = ['mean_residual', 'mean_residual_pct', 'performance_ratio', 'std_residual']

iso_data = inv_daily[anomaly_features].fillna(0)
iso_scaler = MinMaxScaler()
iso_scaled = iso_scaler.fit_transform(iso_data)

iso_forest = IsolationForest(n_estimators=200, contamination=0.1, random_state=42)
inv_daily['anomaly'] = iso_forest.fit_predict(iso_scaled)
inv_daily['anomaly_score'] = iso_forest.score_samples(iso_scaled)
# -1 = anomaly, 1 = normal
inv_daily['is_anomaly'] = inv_daily['anomaly'] == -1

anomaly_pct = inv_daily['is_anomaly'].mean() * 100
print(f'Anomalous inverter-days detected: {inv_daily["is_anomaly"].sum()} / {len(inv_daily)} ({anomaly_pct:.1f}%)')

In [ ]:
# ---- Rank inverters by anomaly frequency ----
inv_anomaly_rank = inv_daily.groupby('SOURCE_KEY').agg(
    anomaly_days=('is_anomaly', 'sum'),
    total_days=('is_anomaly', 'count'),
    mean_perf_ratio=('performance_ratio', 'mean'),
    mean_residual=('mean_residual', 'mean')
).reset_index()
inv_anomaly_rank['anomaly_rate'] = inv_anomaly_rank['anomaly_days'] / inv_anomaly_rank['total_days']
inv_anomaly_rank = inv_anomaly_rank.sort_values('anomaly_rate', ascending=False)

print('Top 5 most anomalous inverters:')
display(inv_anomaly_rank.head())

In [ ]:
# ---- Visualise: Anomaly Rate per Inverter ----
plt.figure(figsize=(14, 5))
colors = ['red' if r > 0.15 else 'steelblue' for r in inv_anomaly_rank['anomaly_rate']]
bars = plt.bar(inv_anomaly_rank['SOURCE_KEY'], inv_anomaly_rank['anomaly_rate'], color=colors)
plt.axhline(0.15, color='orange', linestyle='--', label='Threshold (15%)')
plt.xticks(rotation=90, fontsize=7)
plt.ylabel('Anomaly Rate (fraction of days)')
plt.title('Plant 1 – Inverter Anomaly Rate (Red = High-Risk)', fontsize=13)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ---- Visualise: Performance Ratio vs Anomaly Rate scatter ----
plt.figure(figsize=(8, 6))
scatter = plt.scatter(
    inv_anomaly_rank['mean_perf_ratio'],
    inv_anomaly_rank['anomaly_rate'],
    c=inv_anomaly_rank['anomaly_rate'], cmap='RdYlGn_r', s=80, edgecolors='k', linewidths=0.5
)
plt.colorbar(scatter, label='Anomaly Rate')
plt.axvline(1.0, color='gray', linestyle='--', alpha=0.7, label='Perfect ratio=1.0')
plt.xlabel('Mean Performance Ratio (Actual/Predicted)')
plt.ylabel('Anomaly Rate')
plt.title('Performance Ratio vs Anomaly Rate per Inverter', fontsize=13)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ---- Time-series: Actual vs Predicted for a top anomalous inverter ----
worst_inverter = inv_anomaly_rank.iloc[0]['SOURCE_KEY']
inv_ts = p1_inv_day[p1_inv_day['SOURCE_KEY'] == worst_inverter].copy()

plt.figure(figsize=(14, 5))
plt.plot(inv_ts['DATE_TIME'], inv_ts['AC_POWER'], label='Actual AC Power', color='steelblue', linewidth=1)
plt.plot(inv_ts['DATE_TIME'], inv_ts['AC_PREDICTED'], label='Predicted AC Power', color='orange', linewidth=1, linestyle='--')
anomaly_rows = inv_ts[inv_ts['RESIDUAL_PCT'] < inv_ts['RESIDUAL_PCT'].quantile(0.1)]
plt.scatter(anomaly_rows['DATE_TIME'], anomaly_rows['AC_POWER'], color='red', s=20, zorder=5, label='Underperformance Points')
plt.title(f'Actual vs Predicted – Inverter: {worst_inverter}', fontsize=12)
plt.xlabel('Date')
plt.ylabel('AC Power')
plt.legend()
plt.tight_layout()
plt.show()

### 6b. Anomaly Detection – Plant 2
Repeat the anomaly pipeline on Plant 2 to compare which plant has more hardware issues.

In [ ]:
# ---- Per-inverter anomaly detection for Plant 2 ----
p2['DATE_TIME'] = pd.to_datetime(p2['DATE_TIME'])
p2_inv_weather = pd.merge_asof(
    p2.sort_values('DATE_TIME'),
    sr_15min.sort_values('DATE_TIME'),
    on='DATE_TIME',
    tolerance=pd.Timedelta('15min'),
    direction='nearest'
).dropna(subset=['Radiation'])

p2_inv_weather['hour'] = p2_inv_weather['DATE_TIME'].dt.hour
p2_inv_weather['month'] = p2_inv_weather['DATE_TIME'].dt.month
p2_inv_weather['hour_from_noon'] = (p2_inv_weather['hour'] - 12).abs()
p2_inv_day = p2_inv_weather[p2_inv_weather['Radiation'] > 10].copy()

# Use the tuned model (trained on Plant 1) to predict for Plant 2
p2_inv_day['AC_PREDICTED'] = best_tuned_rf.predict(p2_inv_day[FEATURES])
p2_inv_day['RESIDUAL'] = p2_inv_day['AC_POWER'] - p2_inv_day['AC_PREDICTED']
p2_inv_day['RESIDUAL_PCT'] = p2_inv_day['RESIDUAL'] / (p2_inv_day['AC_PREDICTED'].abs() + 1)
p2_inv_day['date'] = p2_inv_day['DATE_TIME'].dt.date

p2_inv_daily = p2_inv_day.groupby(['SOURCE_KEY', 'date']).agg(
    mean_residual=('RESIDUAL', 'mean'),
    mean_residual_pct=('RESIDUAL_PCT', 'mean'),
    total_AC=('AC_POWER', 'sum'),
    total_AC_pred=('AC_PREDICTED', 'sum'),
    std_residual=('RESIDUAL', 'std')
).reset_index()
p2_inv_daily['performance_ratio'] = p2_inv_daily['total_AC'] / (p2_inv_daily['total_AC_pred'] + 1)

# Isolation Forest on Plant 2
p2_iso_data = p2_inv_daily[anomaly_features].fillna(0)
p2_iso_scaled = iso_scaler.fit_transform(p2_iso_data)  # fit fresh scaler
iso_forest_p2 = IsolationForest(n_estimators=200, contamination=0.1, random_state=42)
p2_inv_daily['anomaly'] = iso_forest_p2.fit_predict(p2_iso_scaled)
p2_inv_daily['is_anomaly'] = p2_inv_daily['anomaly'] == -1

p2_inv_anomaly_rank = p2_inv_daily.groupby('SOURCE_KEY').agg(
    anomaly_days=('is_anomaly', 'sum'),
    total_days=('is_anomaly', 'count'),
    mean_perf_ratio=('performance_ratio', 'mean')
).reset_index()
p2_inv_anomaly_rank['anomaly_rate'] = p2_inv_anomaly_rank['anomaly_days'] / p2_inv_anomaly_rank['total_days']

# Side-by-side anomaly rate comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, rank_df, label, color in zip(axes,
    [inv_anomaly_rank, p2_inv_anomaly_rank],
    ['Plant 1', 'Plant 2'],
    ['steelblue', 'darkorange']):
    rank_sorted = rank_df.sort_values('anomaly_rate', ascending=False)
    bar_colors = ['red' if r > 0.15 else color for r in rank_sorted['anomaly_rate']]
    ax.bar(rank_sorted['SOURCE_KEY'], rank_sorted['anomaly_rate'], color=bar_colors)
    ax.axhline(0.15, color='black', linestyle='--', alpha=0.5, label='15% threshold')
    ax.set_title(f'{label} – Inverter Anomaly Rate')
    ax.set_xticklabels(rank_sorted['SOURCE_KEY'], rotation=90, fontsize=6)
    ax.set_ylabel('Anomaly Rate')
    ax.legend()

plt.suptitle('Plant 1 vs Plant 2 – Inverter Anomaly Rates Compared', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

p1_mean = inv_anomaly_rank['anomaly_rate'].mean()
p2_mean = p2_inv_anomaly_rank['anomaly_rate'].mean()
print(f"Plant 1 mean anomaly rate: {p1_mean:.3f}")
print(f"Plant 2 mean anomaly rate: {p2_mean:.3f}")
print(f"{'Plant 1' if p1_mean > p2_mean else 'Plant 2'} has more anomalous behaviour overall.")

---
## 8. Plant 1 vs Plant 2 – Comparative Analysis

In [ ]:
# ---- Summary statistics comparison ----
summary = pd.DataFrame({
    'Metric': ['Mean AC Power', 'Max AC Power', 'Mean DC Power', 'Total Inverters', 'DC→AC Efficiency'],
    'Plant 1': [
        f"{p1['AC_POWER'].mean():.1f}",
        f"{p1['AC_POWER'].max():.1f}",
        f"{p1['DC_POWER'].mean():.1f}",
        f"{p1['SOURCE_KEY'].nunique()}",
        f"{(p1['AC_POWER'].sum() / (p1['DC_POWER'].sum()+1)):.4f}"
    ],
    'Plant 2': [
        f"{p2['AC_POWER'].mean():.1f}",
        f"{p2['AC_POWER'].max():.1f}",
        f"{p2['DC_POWER'].mean():.1f}",
        f"{p2['SOURCE_KEY'].nunique()}",
        f"{(p2['AC_POWER'].sum() / (p2['DC_POWER'].sum()+1)):.4f}"
    ]
})
display(summary)

In [ ]:
# ---- DC/AC Power distributions ----
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, col, title in zip(axes, ['DC_POWER', 'AC_POWER'], ['DC Power', 'AC Power']):
    # Filter positive values only
    d1 = p1[p1[col] > 0][col]
    d2 = p2[p2[col] > 0][col]
    ax.hist(d1, bins=50, alpha=0.6, label='Plant 1', color='steelblue', density=True)
    ax.hist(d2, bins=50, alpha=0.6, label='Plant 2', color='darkorange', density=True)
    ax.set_title(f'{title} Distribution (Daytime Only)')
    ax.set_xlabel(col)
    ax.set_ylabel('Density')
    ax.legend()

plt.suptitle('Plant 1 vs Plant 2 – Power Output Distributions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 9. Summary & Conclusions

### Problem Defined
Predict AC power generation from weather data (regression), then flag inverters that consistently underperform relative to their weather-adjusted expected yield (anomaly detection). This mimics real operations monitoring systems used in commercial solar farms.

### Regression Results
- **Best model after tuning**: Random Forest with `RandomizedSearchCV` + `TimeSeriesSplit` achieves the highest R² on the holdout test set.
- **Key predictor**: Solar radiation dominates — confirmed by both feature importance and SHAP analysis.
- **SHAP insight**: High humidity consistently pushes predictions down (acts as cloud cover proxy). Hour-of-day captures the sun angle effect.
- **Cross-plant generalization**: Model trained on Plant 1 transfers well to Plant 2, confirming the weather–power relationship is physics-driven, not plant-specific.
- **Time-series CV**: TimeSeriesSplit gives more honest, temporally consistent performance estimates than random split.

### Anomaly Detection Results
- **Isolation Forest** flagged ~10% of inverter-days as anomalous in both plants.
- Inverter×date heatmaps reveal which anomalies are persistent (hardware/degradation = always red for a specific inverter) vs. correlated across all inverters (weather event = whole column red).
- Top anomalous inverters are strong candidates for physical inspection.

### Plant Comparison
- Both plants have 22 inverters. DC→AC conversion efficiency is similar.
- Side-by-side anomaly comparison shows which plant has more operational issues.

### Beyond the Course (New Techniques)
1. **Isolation Forest** — unsupervised anomaly detection
2. **TimeSeriesSplit CV** — temporally correct cross-validation for time-series
3. **RandomizedSearchCV** — efficient hyperparameter tuning
4. **SHAP values** — directional model explainability (feature impact direction, not just magnitude)
5. **merge_asof** — time-aligned joining of multi-frequency time-series datasets
6. **Performance Ratio** — domain-specific KPI used in real solar plant monitoring (IEC 61724 standard)
